# TRELLIS.2 – Text/Image to 3D (Flash Attention 2)
تشغيل مشروع [TRELLIS.2-Text-to-3D-FA2](https://github.com/PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2) على Google Colab.

**تنبيه:** Flash Attention 2 يتطلب معالج Ampere أو أحدث (مثل A100). على T4 المجاني سيتم استخدام الانتباه العادي تلقائياً.
شغّل الخلايا بالتسلسل **دون تخطي**.

In [ ]:
import torch
import subprocess, sys

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("الرجاء تفعيل GPU من Runtime → Change runtime type → GPU")

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"GPU: {gpu_name}")

install_flash = any(x in gpu_name for x in ["A100", "A6000", "3090", "4090", "L4", "L40"])
if install_flash:
    print("✅ GPU يدعم Flash Attention 2")
else:
    print("ℹ️ GPU لا يدعم Flash Attention، سيتم استخدام SDPA (انتباه PyTorch العادي)")

In [ ]:
import os, sys

# استنساخ المستودع إذا لم يكن موجوداً
if not os.path.exists("TRELLIS.2-Text-to-3D-FA2"):
    !git clone https://github.com/PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2.git
%cd TRELLIS.2-Text-to-3D-FA2

# تثبيت spconv (ضروري جداً) – مع معالجة أخطاء التسمية
cuda_ver = torch.version.cuda.replace(".", "")
try:
    !pip install spconv-cu{cuda_ver}
except:
    print("محاولة تثبيت spconv العام...")
    !pip install spconv

# تثبيت المتطلبات الأخرى
!pip install -r requirements.txt 2>/dev/null || echo "لا يوجد requirements.txt"
!pip install trimesh pygltflib viser omegaconf opencv-python imageio[ffmpeg] gradio huggingface_hub 2>&1 | tail -5

# تثبيت Flash Attention إذا كان مدعوماً
if install_flash:
    !pip install ninja packaging
    !pip install flash-attn --no-build-isolation 2>&1 | tail -5
    print("✅ Flash Attention 2 تم تثبيته")
else:
    print("⏩ تخطي تثبيت flash-attn")

print("✅ تم تثبيت جميع المتطلبات")

In [ ]:
import sys, os

# إضافة المستودع إلى مسار Python (بديل عن pip install -e .)
repo_path = "/content/TRELLIS.2-Text-to-3D-FA2"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# استيراد خط الأنابيب
import torch
from trellis.pipelines import TrellisImageTo3DPipeline

# تحميل النموذج
pipeline = TrellisImageTo3DPipeline.from_pretrained(
    "JeffreyXiang/TRELLIS-image-large",
    torch_dtype=torch.float16
)
pipeline.to("cuda")
print("✅ تم تحميل خط الأنابيب بنجاح")

In [ ]:
from PIL import Image
import requests

# تحميل صورة تجريبية (استبدلها بصورتك إذا أردت)
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/tiger.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
display(image)

In [ ]:
# توليد المجسم الثلاثي الأبعاد
outputs = pipeline.run(
    image,
    seed=42
)
print("🔹 تم الانتهاء من التوليد")
print("المفاتيح الناتجة:", outputs.keys())
gaussian = outputs.get('gaussian')
mesh = outputs.get('mesh')

In [ ]:
import numpy as np
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# تصدير الشبكة (mesh) بصيغة GLB
if mesh is not None:
    glb_path = out_dir / "model.glb"
    mesh.export(glb_path)
    print(f"✅ تم تصدير الشبكة إلى {glb_path}")
else:
    print("⚠️ لا توجد شبكة في المخرجات")

# تصدير السحابة الغاوسية بصيغة PLY (إن احتجتها)
if gaussian is not None:
    ply_path = out_dir / "gaussian.ply"
    # بعض الإصدارات توفر دالة save_ply
    if hasattr(gaussian[0], 'save_ply'):
        gaussian[0].save_ply(ply_path)
        print(f"✅ تم تصدير Gaussians إلى {ply_path}")
    else:
        # تصدير بسيط للنقاط بصيغة PLY يدوياً
        from trellis.utils.ply import save_ply  # إن وُجد
        try:
            save_ply(ply_path, gaussian[0].xyz, gaussian[0].features_dc)
            print(f"✅ تم تصدير Gaussians يدوياً إلى {ply_path}")
        except Exception as e:
            print(f"⚠️ تعذر تصدير PLY: {e}")

In [ ]:
# إعداد العارض التفاعلي باستخدام viser + منفذ Colab الداخلي (بدون ngrok)
import viser
from google.colab.output import eval_js
from IPython.display import IFrame, display
import time

# تشغيل خادم viser على منفذ 8080
server = viser.ViserServer(host="0.0.0.0", port=8080)

# إضافة المحتوى إلى المشهد
if gaussian is not None and len(gaussian) > 0:
    try:
        # استخدام دالة العرض المدمجة إن وُجدت
        from trellis.utils import render_utils
        render_utils.render_gaussian(server, gaussian[0])
        print("✅ تم عرض السحابة الغاوسية")
    except ImportError:
        print("⚠️ تعذر استيراد render_utils، جارٍ إضافة سحابة نقاط بسيطة...")
        # إضافة سحابة نقاط بسيطة (بدون ألوان معقدة)
        points = gaussian[0].xyz.detach().cpu().numpy().reshape(-1, 3)
        server.scene.add_point_cloud(
            "/points",
            points=points,
            colors=(255, 200, 200),
            point_size=0.01
        )
        print("✅ تم إضافة سحابة نقاط مبدئية")
else:
    print("لا توجد بيانات غاوسية للعرض")

# إنشاء رابط الوكيل من Colab
url = eval_js(f"google.colab.kernel.proxyPort(8080)")
public_url = f"https://{url}"
print(f"🔗 رابط المشاهدة التفاعلية: {public_url}")

# عرض العارض داخل دفتر الملاحظات
display(IFrame(src=public_url, width="100%", height="600px"))
print("إذا لم يظهر المشهد، افتح الرابط أعلاه في نافذة جديدة.")

## ملاحظات ختامية
- تم تجنب `pip install -e .` واستخدمنا `sys.path` مما يحل مشكلة `ModuleNotFoundError`.
- العارض التفاعلي يعمل عبر منفذ Colab الداخلي **بدون الحاجة إلى ngrok**.
- الملفات المصدرة `model.glb` و `gaussian.ply` موجودة في مجلد `outputs` يمكن تحميلها.
- لتغيير الصورة، عدّل رابط الصورة في خلية `load-image` أو ارفع صورتك الخاصة.